In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

Path("../outputs/metrics").mkdir(parents=True, exist_ok=True)
Path("../outputs/models").mkdir(parents=True, exist_ok=True)

cpu


In [3]:
with open("../data/graph/train_graph.pkl", "rb") as f:
    train_graph = pickle.load(f)

with open("../data/graph/test_graph.pkl", "rb") as f:
    test_graph = pickle.load(f)

with open("../data/graph/wallet_mapping.pkl", "rb") as f:
    wallet_mapping = pickle.load(f)

num_nodes = wallet_mapping["num_nodes"]

print("Num nodes:", num_nodes)
print("Train:", train_graph["edge_features"].shape)
print("Test:", test_graph["edge_features"].shape)

Num nodes: 108582
Train: (38454, 10)
Test: (298160, 10)


In [4]:
class EdgeDataset(Dataset):
    def __init__(self, graph):
        self.edge_index = torch.tensor(graph["edge_index"], dtype=torch.long)
        self.edge_features = torch.tensor(graph["edge_features"], dtype=torch.float32)
        self.labels = torch.tensor(graph["edge_labels"], dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        source = self.edge_index[0, idx]
        target = self.edge_index[1, idx]
        edge_feat = self.edge_features[idx]
        label = self.labels[idx]
        return source, target, edge_feat, label

In [5]:
BATCH_SIZE = 1024

train_dataset = EdgeDataset(train_graph)
test_dataset = EdgeDataset(test_graph)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [6]:
class GraphEdgeClassifier(nn.Module):
    def __init__(self, num_nodes, edge_feat_dim, embedding_dim=64, hidden_dim=128):
        super().__init__()

        self.node_embedding = nn.Embedding(num_nodes, embedding_dim)

        input_dim = embedding_dim * 2 + edge_feat_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, source, target, edge_feat):
        src_emb = self.node_embedding(source)
        tgt_emb = self.node_embedding(target)

        x = torch.cat([src_emb, tgt_emb, edge_feat], dim=1)
        logits = self.mlp(x).squeeze(1)

        return logits

In [7]:
edge_feat_dim = train_graph["edge_features"].shape[1]

model = GraphEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat_dim,
    embedding_dim=64,
    hidden_dim=128
).to(DEVICE)

In [8]:
criterion = nn.BCEWithLogitsLoss()

In [9]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [10]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()

        logits = model(source, target, edge_feat)
        loss = criterion(logits, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [11]:
def evaluate(model, loader, threshold=0.3):
    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for source, target, edge_feat, label in loader:
            source = source.to(DEVICE)
            target = target.to(DEVICE)
            edge_feat = edge_feat.to(DEVICE)

            logits = model(
                source,
                target,
                edge_feat
            )

            probs = torch.sigmoid(logits)

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_labels.extend(
                label.numpy()
            )

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    preds = (all_probs >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(all_labels, preds),
        "precision": precision_score(all_labels, preds, zero_division=0),
        "recall": recall_score(all_labels, preds, zero_division=0),
        "f1": f1_score(all_labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "pr_auc": average_precision_score(all_labels, all_probs),
        "confusion_matrix": confusion_matrix(all_labels, preds)
    }

    return metrics

In [12]:
EPOCHS = 20

history = []
best_f1 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_metrics = evaluate(model, test_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "roc_auc": test_metrics["roc_auc"],
        "pr_auc": test_metrics["pr_auc"]
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {test_metrics['accuracy']:.4f} | "
        f"Prec: {test_metrics['precision']:.4f} | "
        f"Rec: {test_metrics['recall']:.4f} | "
        f"F1: {test_metrics['f1']:.4f} | "
        f"ROC-AUC: {test_metrics['roc_auc']:.4f} | "
        f"PR-AUC: {test_metrics['pr_auc']:.4f}"
    )

    if test_metrics["f1"] > best_f1:
        best_f1 = test_metrics["f1"]

        torch.save(
            model.state_dict(),
            "../outputs/models/graph_plain_best.pt"
        )

Epoch 01 | Loss: 0.3131 | Acc: 0.9998 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: 0.5363 | PR-AUC: 0.0002
Epoch 02 | Loss: 0.0916 | Acc: 0.9998 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: 0.6574 | PR-AUC: 0.0007
Epoch 03 | Loss: 0.0804 | Acc: 0.9998 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: 0.6807 | PR-AUC: 0.0011
Epoch 04 | Loss: 0.0752 | Acc: 0.9998 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: 0.6858 | PR-AUC: 0.0017
Epoch 05 | Loss: 0.0724 | Acc: 0.9998 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: 0.6895 | PR-AUC: 0.0031
Epoch 06 | Loss: 0.0693 | Acc: 0.9998 | Prec: 0.1250 | Rec: 0.0156 | F1: 0.0278 | ROC-AUC: 0.6955 | PR-AUC: 0.0078
Epoch 07 | Loss: 0.0675 | Acc: 0.9997 | Prec: 0.1364 | Rec: 0.0469 | F1: 0.0698 | ROC-AUC: 0.6909 | PR-AUC: 0.0315
Epoch 08 | Loss: 0.0637 | Acc: 0.9997 | Prec: 0.1111 | Rec: 0.0625 | F1: 0.0800 | ROC-AUC: 0.6892 | PR-AUC: 0.0540
Epoch 09 | Loss: 0.0600 | Acc: 0.9995 | Prec: 0.0460 | Rec: 0.0625 | F1: 0.0530 

In [13]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "../outputs/metrics/graph_plain_history.csv",
    index=False
)

history_df

,epoch,train_loss,accuracy,precision,recall,f1,roc_auc,pr_auc
0,1,0.313067,0.999785,0.000000,0.000000,0.000000,0.536292,0.000234
1,2,0.091608,0.999785,0.000000,0.000000,0.000000,0.657411,0.000703
2,3,0.080420,0.999785,0.000000,0.000000,0.000000,0.680742,0.001139
3,4,0.075171,0.999785,0.000000,0.000000,0.000000,0.685799,0.001715
4,5,0.072354,0.999775,0.000000,0.000000,0.000000,0.689456,0.003142
5,6,0.069320,0.999765,0.125000,0.015625,0.027778,0.695506,0.007807
6,7,0.067461,0.999732,0.136364,0.046875,0.069767,0.690856,0.031465
7,8,0.063671,0.999691,0.111111,0.062500,0.080000,0.689199,0.053980
8,9,0.059988,0.999520,0.045977,0.062500,0.052980,0.685536,0.063092
9,10,0.056212,0.999162,0.020619,0.062500,0.031008,0.683565,0.063078


In [14]:
model.load_state_dict(
    torch.load("../outputs/models/graph_plain_best.pt")
)

final_metrics = evaluate(model, test_loader)

print("Final Metrics:")
for k, v in final_metrics.items():
    if k != "confusion_matrix":
        print(k, ":", v)

print("Confusion Matrix:")
print(final_metrics["confusion_matrix"])

Final Metrics:
threshold : 0.3
accuracy : 0.9996914408371345
precision : 0.1111111111111111
recall : 0.0625
f1 : 0.08
roc_auc : 0.6891993791429606
pr_auc : 0.053979804509767526
Confusion Matrix:
[[298064     32]
 [    60      4]]


In [15]:
final_result = {
    "model": "graph_plain",
    "accuracy": final_metrics["accuracy"],
    "precision": final_metrics["precision"],
    "recall": final_metrics["recall"],
    "f1": final_metrics["f1"],
    "roc_auc": final_metrics["roc_auc"],
    "pr_auc": final_metrics["pr_auc"],
    "tn": final_metrics["confusion_matrix"][0, 0],
    "fp": final_metrics["confusion_matrix"][0, 1],
    "fn": final_metrics["confusion_matrix"][1, 0],
    "tp": final_metrics["confusion_matrix"][1, 1],
}

pd.DataFrame([final_result]).to_csv(
    "../outputs/metrics/graph_plain_final.csv",
    index=False
)

final_result

{'model': 'graph_plain',
 'accuracy': 0.9996914408371345,
 'precision': 0.1111111111111111,
 'recall': 0.0625,
 'f1': 0.08,
 'roc_auc': 0.6891993791429606,
 'pr_auc': 0.053979804509767526,
 'tn': np.int64(298064),
 'fp': np.int64(32),
 'fn': np.int64(60),
 'tp': np.int64(4)}